Постановка задачи (генерация синтетического датасета)

In [ ]:
"""
# Спецификация синтетического датасета субъектов кредитных историй

### 0. Объём данных

- Всего: **1 000 000 строк**.

---

### 1. Структура полей

- `source` — `string`
- `id_sub` — `longint` (идентификатор субъекта, одно физическое лицо)
- `surname` — `string` (фамилия)
- `name` — `string` (имя)
- `secondname` — `string` (отчество)
- `birthdate` — `date` (дата рождения)
- `INN` — `string`
- `SNILS` — `string`
- `docnum` — `string` (номер документа)
- `docseries` — `string` (серия документа)

---

### 2. Источник данных

- Для всех записей:
  `source = "GPT"`.

---

### 3. Этнокультурное происхождение и пол

- **Происхождение ФИО:**
  - 80% — восточнославянского происхождения;
  - 19% — среднеазиатского происхождения (могут не иметь отчества);
  - 1% — иные страны.
- **Пол:**
  - мужчины / женщины в пропорции **50 / 50**.

---

### 4. Распределение записей по `id_sub`

- Каждый `id_sub` соответствует **отдельному лицу**.
- На каждое лицо приходится от **1 до 15** разнородных записей.
- Среднее количество записей — **≈ 5** на `id_sub`.
- Для **невосточнославянских** лиц, как правило, записей **больше**,
  так как вероятность ошибок и разнородной информации выше.

---

### 5. Разнородность записей по одному `id_sub`

Для одного и того же `id_sub`:

- В **80%** случаев разнородная запись отличается **ровно в одном** из полей:
  - `surname`, `name`, `secondname`,
    `birthdate`, `INN`, `SNILS`, `docnum`, `docseries`.
- В **20%** случаев запись отличается **в нескольких** из этих полей.

---

### 6. Кириллица / латиница

Разнородные записи включают:

- ФИО, записанные **на кириллице**;
- ФИО, записанные **на латинице**;
- комбинации (смешивание написаний по одному и тому же лицу).

Особенности:

- Проблема смешения кириллицы и латиницы **в 80% случаев** относится к
  **невосточнославянским** заёмщикам.

---

### 7. Наличие / отсутствие отчества

Разнородные записи по одному `id_sub` могут включать:

- Вариант **с отчеством**;
- Вариант **без отчества**.

При этом:

- В **80% случаев** такая проблема характерна для **невосточнославянских** лиц.

---

### 8. Ошибки и вариации даты рождения

Разнородные записи включают:

- Полностью **отсутствующую** дату рождения;
- Ошибки по дате рождения:
  - **± 1–2 дня**;
  - **± 1–2 месяца**;
  - **± 1 год**.

**Пример набора вариаций одной даты:**

- `19.09.1975`
- `18.09.1975`
- `19.09.1976`
- `19.07.1975`

---

### 9. Перепутанные местами части ФИО

Разнородные записи по одному лицу могут включать:

- Перепутанные местами **имя и фамилия**;
- Перепутанные **фамилия и отчество**.

---

### 10. Опечатки в ФИО (расстояние Левенштейна)

Разнородные записи включают:

- Ошибки в одном из полей ФИО (`surname`, `name`, `secondname`);
- Расстояние Левенштейна **1–2** символа;
- Часто ошибки соответствуют **соседним клавишам русской раскладки**
  (ошибки из-за промаха по русской клавиатуре).

---

### 11. Пробельные и мусорные значения

Разнородные записи могут включать:

- **Пробельные значения:**
  - одиночный пробел;
  - табуляция.
- **Мусорные маркеры**, например:
  - `-`, `_`;
  - `отсутствует`, `неизвестно`;
  - `Б/Н`, `н/д`;
  - `N/A`;
  - `0`, `000000`, `1111111` и подобные.

Эти значения могут встречаться в:

- полях ФИО (`surname`, `name`, `secondname`),
- идентификаторах (`INN`, `SNILS`, `docnum`, `docseries`).

---

### 12. Неправильное разделение фамилии (невосточнославянские лица)

Для невосточнославянских лиц разнородные записи включают:

- Неправильное разделение фамилии и имени, например:
  - `мухамеджон магомедов`;
  - `мухамед джонмагомедов`;
  - и иные вариации, где части ФИО разделены/слеплены некорректно.

---

### 13. Созвучные имена и фамилии

Разнородные записи включают:

- Созвучные фамилии и имена, например:
  - `Карелин / Кореев / Корнеев`;
  - другие сочетания близко звучащих ФИО.

---

### 14. Коллизии идентификаторов между разными лицами

В **1% случаев**:

- Встречаются записи, где:
  - **ИНН**, **СНИЛС**, паспортные данные (`docnum`, `docseries`) **совпадают**,
  - но **ФИО различаются**, и это **разные заёмщики** (разные `id_sub`).

---

### 15. Слитные и раздельные серия/номер документа

Разнородные записи включают:

- Ситуацию, когда:
  - в одной записи `docseries` и `docnum` хранятся **вместе в одном поле**
    (например, только в `docnum`);
  - в другой записи по тому же лицу они **заполнены раздельно**
    (`docseries` и `docnum` отдельно).

---

### 16. Совпадающие ФИО у разных лиц

Предусматривается:

- Для популярных имён и фамилий возможны полные совпадения ФИО:
  - одинаковые `surname`, `name`, `secondname`;
  - при необходимости — и `birthdate`.
- При этом это **разные лица** (разные `id_sub`).

---

### 17. Изменение фамилии у женщин

Разнородные записи включают:

- Изменение фамилии у женщин при смене фамилии после замужества:
  - одна женщина может выйти замуж **дважды**;
  - по одному `id_sub` может быть **до трёх разных фамилий**.

---

### 19. Ошибки в СНИЛС, паспорте и ИНН

Для полей `SNILS`, `docnum`, `docseries`, `INN`:

- Возможны:
  - ошибки с расстоянием Левенштейна **1–2**;
  - реже — **мусорные значения** (см. пункт 11).
- Внутри **одной однородной записи** несколько типов ошибок **одновременно не встречаются**
  (например, не совмещаются и Левенштейн-ошибка, и мусорный маркер для одного и того же поля).

---

### 20. Качество заполнения идентификаторов vs. ФИО

Характеристики качества данных:

- Поля `SNILS`, `INN`, паспортные реквизиты заполнены **значительно хуже**, чем:
  - ФИО (`surname`, `name`, `secondname`);
  - дата рождения (`birthdate`).
- Для идентификаторов:
  - существенно чаще встречаются **мусорные значения**;
  - существенно чаще встречаются **пропуски**, особенно для **невосточнославянских** лиц.
- Почти по каждому `id_sub`:
  - есть **пропуски хотя бы в одном** из полей: `INN`, `SNILS`, `docseries`.

При этом:

- `docnum` **никогда не бывает пустым**,
  но может содержать **мусорные** значения.
- `docseries` пустой примерно в **3%** записей.
- `INN` пустой примерно в **58%** записей.
- `SNILS` пустой примерно в **62%** записей.

ВЕРСИЯ 1 ОТЛИЧАЕТСЯ ОТ ВЕРСИЯ 2
Новая колонка id_loan, которая хорошо влияет на дедупликацию

Появилась структура “субъект–договор”:

~90% договоров с одним заёмщиком (1 id_sub на id_loan),

~5% субъектов участвуют в договорах с 2+ заёмщиками (один id_loan на несколько id_sub),

есть ошибки недодубликации: один id_sub может иметь несколько разных id_loan, которые логически один договор.
"""


'\n# Спецификация синтетического датасета субъектов кредитных историй\n\n### 0. Объём данных\n\n- Всего: **1 000 000 строк**.\n\n---\n\n### 1. Структура полей\n\n- `source` — `string`\n- `id_sub` — `longint` (идентификатор субъекта, одно физическое лицо)\n- `surname` — `string` (фамилия)\n- `name` — `string` (имя)\n- `secondname` — `string` (отчество)\n- `birthdate` — `date` (дата рождения)\n- `INN` — `string`\n- `SNILS` — `string`\n- `docnum` — `string` (номер документа)\n- `docseries` — `string` (серия документа)\n\n---\n\n### 2. Источник данных\n\n- Для всех записей:  \n  `source = "GPT"`.\n\n---\n\n### 3. Этнокультурное происхождение и пол\n\n- **Происхождение ФИО:**\n  - 80% — восточнославянского происхождения;\n  - 19% — среднеазиатского происхождения (могут не иметь отчества);\n  - 1% — иные страны.\n- **Пол:**\n  - мужчины / женщины в пропорции **50 / 50**.\n\n---\n\n### 4. Распределение записей по `id_sub`\n\n- Каждый `id_sub` соответствует **отдельному лицу**.\n- На каждое ли

Версия датасета 1 (без id_договоров)

In [ ]:

# -*- coding: utf-8 -*-
import random, csv, math, os, sys
from datetime import datetime, timedelta
from typing import Iterator, Tuple, Dict, List
try:
    from openpyxl import Workbook
except Exception:
    Workbook = None  # allow CSV fallback if openpyxl not present

# ---------------- CONFIG ----------------
TOTAL_ROWS = 1_000_000
AVG_ROWS_PER_PERSON = 5.1
OUT_PATH = "credit_subjects_1M.xlsx"  # change to .csv for CSV output
RANDOM_SEED = 20251014

INN_EMPTY_RATE = 0.58
SNILS_EMPTY_RATE = 0.62
DOCSER_EMPTY_RATE = 0.03

# Mixing format share (series+number in one field)
COMBINE_RATE = 0.35

random.seed(RANDOM_SEED)

# ---------------- Static resources ----------------
garbage_vals = ["", " ", "\t", "-", "_", "отсутствует", "неизвестно", "Б/Н", "н/д", "N/A", "0", "000000", "1111111"]
male_first_ru = ["Александр","Сергей","Дмитрий","Андрей","Алексей","Иван","Николай","Павел","Михаил","Евгений",
                 "Владимир","Константин","Григорий","Ярослав","Олег","Тимофей","Кирилл","Фёдор","Роман","Егор"]
female_first_ru = ["Анна","Елена","Ольга","Наталья","Татьяна","Мария","Ирина","Алина","Екатерина","Светлана",
                   "Дарья","Виктория","Юлия","Вероника","Ксения","Анастасия","Полина","Людмила","София","Валерия"]
surnames_ru = ["Иванов","Петров","Сидоров","Смирнов","Кузнецов","Попов","Соколов","Лебедев","Козлов","Новиков",
               "Морозов","Ершов","Орлов","Волков","Соловьёв","Егоров","Фролов","Васильев","Зайцев","Карелин",
               "Кореев","Корнеев","Громов","Калинин","Макаров","Никитин","Поляков","Тимофеев","Федоров","Яковлев"]
male_first_cas = ["Алишер","Мурад","Давран","Шерали","Жомарт","Ислом","Шухрат","Улукбек","Абдурахман","Магомед",
                  "Мухаммад","Рустам","Фарход","Карим","Нурлан","Сардор","Бахтиёр","Эркин","Азамат","Тимир"]
female_first_cas = ["Азиза","Гульнара","Дилноза","Мехринисо","Ширин","Нилуфар","Саодат","Сания","Зухра","Мунира",
                    "Севара","Динара","Айгерим","Айдана","Айгуль","Мадина","Лейла","Гульзира","Гульмира","Жазира"]
surnames_cas = ["Каримов","Нурматов","Ибрагимов","Нуралиев","Хашимов","Рахмонов","Магомедов","Сафаров","Шарипов","Садыков",
                "Умаров","Азимов","Исмаилов","Турсунов","Усманов","Султанов","Юсупов","Нурмухамедов","Тлеубергенов","Алиев"]
male_first_other = ["John","Michael","David","Daniel","James","Robert","Thomas","William","Richard","Joseph"]
female_first_other = ["Emily","Sophia","Olivia","Emma","Ava","Mia","Chloe","Isabella","Charlotte","Amelia"]
surnames_other = ["Smith","Johnson","Williams","Brown","Jones","Miller","Davis","Wilson","Moore","Taylor"]
patro_roots_m = ["Александров","Сергеев","Дмитриев","Андреев","Алексеев","Иванов","Николаев","Павлов","Михайлов","Евгеньев",
                 "Владимиров","Константинов","Григорьев","Ярославов","Олегов","Тимофеев","Кириллов","Фёдоров","Романов","Егоров"]
patro_male = [root+"ич" for root in patro_roots_m]
patro_female = [root+"на" for root in patro_roots_m]

# ПОХОЖИЕ ФАМИЛИИ
similar_groups = [["Карелин","Кореев","Корнеев"],["Смирнов","Смирнова","Смирной"],["Иванов","Иванова","Иванёва"]]

# ТРАНСЛИТЕРАЦИЯ
translit_map = {"А":"A","Б":"B","В":"V","Г":"G","Д":"D","Е":"E","Ё":"E","Ж":"Zh","З":"Z","И":"I","Й":"Y","К":"K","Л":"L","М":"M",
                "Н":"N","О":"O","П":"P","Р":"R","С":"S","Т":"T","У":"U","Ф":"F","Х":"Kh","Ц":"Ts","Ч":"Ch","Ш":"Sh","Щ":"Sch",
                "Ъ":"","Ы":"Y","Ь":"","Э":"E","Ю":"Yu","Я":"Ya","а":"a","б":"b","в":"v","г":"g","д":"d","е":"e","ё":"e","ж":"zh","з":"z",
                "и":"i","й":"y","к":"k","л":"l","м":"m","н":"n","о":"o","п":"p","р":"r","с":"s","т":"t","у":"u","ф":"f","х":"kh","ц":"ts",
                "ч":"ch","ш":"sh","щ":"sch","ъ":"","ы":"y","ь":"","э":"e","ю":"yu","я":"ya"}
def to_latin(s: str) -> str: return "".join(translit_map.get(ch, ch) for ch in s)

# СОСЕДНИЕ БУКВЫ ПО КЛАВИАТУРЕ
neighbors = {"ф":"ы","ы":"фва","в":"ыас","а":"впс","п":"аро","р":"пош","о":"ршл","л":"одж","д":"лор","ж":"лэ","э":"ж",
             "й":"ц","ц":"йу","у":"цк","к":"уе","е":"кн","н":"ег","г":"нш","ш":"гщ","щ":"шз","з":"щх","х":"зъ","ъ":"х",
             "с":"ама","м":"с","и":"т","т":"и","ь":"б","б":"ью","ю":"б"}
def typo_neighbor(s: str) -> str:
    if not s: return s
    idx = random.randrange(len(s)); ch = s[idx]; repl = ch
    if ch.lower() in neighbors and neighbors[ch.lower()]:
        repl = random.choice(neighbors[ch.lower()]); repl = repl.upper() if ch.isupper() else repl
    return s[:idx] + repl + s[idx+1:]
def typo_lev2(s: str) -> str:
    if not s: return s
    if random.random() < 0.5: return typo_neighbor(typo_neighbor(s))
    else:
        if len(s)>2 and random.random()<0.5:
            i = random.randrange(len(s)); return s[:i] + s[i+1:]
        else:
            i = random.randrange(len(s)); ch = s[i]; return s[:i] + ch + s[i:]

def random_date(start_year=1940, end_year=2005):
    d0 = datetime(start_year,1,1); d1 = datetime(end_year,12,31)
    delta = (d1 - d0).days; return d0 + timedelta(days=random.randrange(delta+1))
def fmt_date(d: datetime) -> str: return d.strftime("%d.%m.%Y")
def perturb_date(d: datetime) -> str:
    choice = random.random()
    if choice < 0.15: return random.choice(garbage_vals)
    if choice < 0.45:
        dd = random.choice([-2,-1,1,2]); return fmt_date(d + timedelta(days=dd))
    elif choice < 0.8:
        mm = random.choice([-2,-1,1,2]); y = d.year; m = max(1, min(12, d.month + mm)); day = min(d.day, 28)
        return f"{day:02d}.{m:02d}.{y:04d}"
    else:
        yy = random.choice([-1,1]); day = min(d.day, 28); return f"{day:02d}.{d.month:02d}.{d.year+yy:04d}"
def gen_inn(): return "".join(str(random.randrange(10)) for _ in range(12))
def gen_snils():
    nums = [random.randrange(10) for _ in range(11)]
    return f"{nums[0]}{nums[1]}{nums[2]}-{nums[3]}{nums[4]}{nums[5]}-{nums[6]}{nums[7]}{nums[8]} {nums[9]}{nums[10]}"
def gen_passport_pair():
    series = f"{random.randrange(1000,10000)}"; number = f"{random.randrange(100000,1000000)}"; return series, number
def combine_series_num(series, num):
    options = [f"{series} {num}", f"{series}{num}", f"{series}-{num}", f"{series}/{num}"]; return random.choice(options)

def sample_record_count(group):
    r = random.random()
    if group == "east":
        return 3 if r<0.15 else 4 if r<0.40 else 5 if r<0.70 else 6 if r<0.90 else 7
    elif group == "central":
        return 4 if r<0.10 else 5 if r<0.30 else 6 if r<0.60 else 7 if r<0.85 else 8 if r<0.95 else 9
    else:
        return 5 if r<0.10 else 6 if r<0.35 else 7 if r<0.65 else 8 if r<0.85 else 9 if r<0.95 else 10

def make_feminine(last):
    if not last.endswith("а") and last.endswith(("ов","ев","ин")): return last + "а"
    return last

def iter_persons(total_rows: int, avg_per_person: float):
    persons_target = int(round(total_rows / avg_per_person))
    east_q = int(round(persons_target*0.80)); central_q = int(round(persons_target*0.19)); other_q = persons_target - east_q - central_q
    male_q = persons_target // 2; female_q = persons_target - male_q

    def split_gender(q, m_rem, f_rem):
        m = min(m_rem, q//2); f = min(f_rem, q-m)
        if f < q-m: need = (q-m) - f; m = min(m+need, m_rem); f = q-m
        return m, f

    m_rem, f_rem = male_q, female_q
    e_m, e_f = split_gender(east_q, m_rem, f_rem); m_rem -= e_m; f_rem -= e_f
    c_m, c_f = split_gender(central_q, m_rem, f_rem); m_rem -= c_m; f_rem -= c_f
    o_m, o_f = split_gender(other_q, m_rem, f_rem); m_rem -= o_m; f_rem -= o_f

    meta = [("east","M")]*e_m + [("east","F")]*e_f + [("central","M")]*c_m + [("central","F")]*c_f + [("other","M")]*o_m + [("other","F")]*o_f
    random.shuffle(meta)

    ks = [sample_record_count(g) for g,_ in meta]
    delta = total_rows - sum(ks)
    idx_e = [i for i,(g,_) in enumerate(meta) if g=="east"]
    idx_c = [i for i,(g,_) in enumerate(meta) if g=="central"]
    idx_o = [i for i,(g,_) in enumerate(meta) if g=="other"]
    while delta != 0:
        if delta > 0:
            pool = idx_c + idx_o + idx_e; random.shuffle(pool)
            for i in pool:
                if ks[i] < 15:
                    ks[i]+=1; delta-=1;
                    if delta==0: break
        else:
            pool = idx_e + idx_c + idx_o; random.shuffle(pool)
            for i in pool:
                if ks[i] > 1:
                    ks[i]-=1; delta+=1;
                    if delta==0: break
    base_id = 90_000_000_000
    for pid, ((gid,sex), k) in enumerate(zip(meta, ks)):
        if gid=="east":
            first = random.choice(male_first_ru if sex=="M" else female_first_ru)
            last = random.choice(surnames_ru)
            if sex=="F": last = make_feminine(last)
            patronymic = random.choice(patro_male if sex=="M" else patro_female)
        elif gid=="central":
            first = random.choice(male_first_cas if sex=="M" else female_first_cas)
            last = random.choice(surnames_cas)
            patronymic = "" if random.random()<0.7 else random.choice(patro_male if sex=="M" else patro_female)
        else:
            first = random.choice(male_first_other if sex=="M" else female_first_other)
            last = random.choice(surnames_other); patronymic = ""
        married_surnames = []
        if sex=="F":
            src_pool = surnames_ru if gid=="east" else (surnames_cas if gid=="central" else surnames_other)
            max_changes = 2 if random.random()<0.35 else 1
            if gid=="east" and random.random()<0.30:
                married_surnames = [make_feminine(random.choice(src_pool)) for _ in range(max_changes)]
            elif gid=="central" and random.random()<0.20:
                married_surnames = [random.choice(src_pool) for _ in range(max_changes)]
            elif gid=="other" and random.random()<0.10:
                married_surnames = [random.choice(src_pool) for _ in range(max_changes)]
            married_surnames = list(dict.fromkeys(married_surnames))
        bdate = random_date(); inn = gen_inn(); snils = gen_snils(); ds, dn = gen_passport_pair()
        base = {"group":gid, "gender":sex, "surname":last, "name":first, "secondname":patronymic,
                "birthdate":fmt_date(bdate), "INN":inn, "SNILS":snils, "docseries":ds, "docnum":dn,
                "married_surnames":married_surnames}
        yield base_id+pid, base, k

def random_name_permutation(surname, name, secondname):
    if random.random() < 0.5: return name, surname, secondname
    else: return (secondname or surname), name, (surname if secondname else "")

identifier_fields = {"INN","SNILS","docnum","docseries"}
field_weights = {"surname":0.15,"birthdate":0.15,"name":0.10,"secondname":0.10,"INN":0.15,"SNILS":0.15,"docnum":0.075,"docseries":0.075}
def weighted_field_choices(k: int):
    fields = list(field_weights.keys()); probs = [field_weights[f] for f in fields]
    # choose k unique according to weights
    chosen = []
    while len(chosen) < k:
        cand = random.choices(fields, weights=probs, k=1)[0]
        if cand not in chosen:
            chosen.append(cand)
        if len(chosen) == len(fields): break
    chosen_ids = [f for f in chosen if f in identifier_fields]
    if len(chosen_ids)>1:
        keep = random.choice(chosen_ids); chosen = [f for f in chosen if f not in identifier_fields or f==keep]
    return chosen

def apply_variations(base):
    gid = base["group"]; sex = base["gender"]
    rec = {k: base[k] for k in ["surname","name","secondname","birthdate","INN","SNILS","docnum","docseries"]}
    multi = (random.random() < 0.20); m = 1 if not multi else random.randint(2,4)
    change_fields = set(weighted_field_choices(m))
    cyr_lat_issue = (gid != "east" and random.random() < 0.8)
    secondname_issue = (gid != "east" and random.random() < 0.8)
    for f in list(change_fields):
        if f in ["surname","name","secondname"]:
            val = rec[f]
            garbage_boost = 0.05 + (0.05 if gid != "east" else 0.0)
            if random.random() < garbage_boost:
                rec[f] = random.choice(garbage_vals); continue
            if f=="surname" and random.random() < 0.15:
                rec[f] = random.choice(random.choice(similar_groups))
            else:
                rec[f] = typo_neighbor(val) if random.random()<0.6 else typo_lev2(val)
        elif f=="birthdate":
            bd = datetime.strptime(base["birthdate"], "%d.%m.%Y"); rec["birthdate"] = perturb_date(bd)
        elif f in identifier_fields:
            if random.random() < 0.15:
                g = random.choice(garbage_vals[1:]) if f=="docnum" else random.choice(garbage_vals)
                if f=="docnum" and g.strip()=="": g="000000"
                rec[f] = g
            else:
                s = rec[f]; digits_idx = [i for i,ch in enumerate(s) if ch.isdigit()]
                if digits_idx:
                    edits = 2 if random.random() < 0.4 else 1
                    s_list = list(s)
                    for _ in range(edits):
                        i = random.choice(digits_idx); s_list[i] = str(random.randrange(10))
                    rec[f] = "".join(s_list)
    if random.random() < 0.10:
        s, n, p = random_name_permutation(rec["surname"], rec["name"], rec["secondname"])
        rec["surname"], rec["name"], rec["secondname"] = s, n, p
    if sex=="F" and base.get("married_surnames"):
        if "surname" in change_fields or random.random()<0.20:
            rec["surname"] = random.choice(base["married_surnames"])
    if cyr_lat_issue:
        for k in ["surname","name"] + (["secondname"] if rec["secondname"] else []):
            if rec[k] and random.random()<0.5:
                rec[k] = to_latin(rec[k])
    if secondname_issue and random.random()<0.5:
        rec["secondname"] = random.choice([rec["secondname"], "", random.choice(garbage_vals)])
    if gid!="east" and random.random()<0.08:
        if rec["surname"] and rec["name"]:
            if random.random()<0.5:
                rec["name"] = rec["name"] + rec["surname"][:2]
            else:
                if len(rec["surname"])>6:
                    cut = random.randint(3, len(rec["surname"])-3)
                    rec["secondname"] = rec["surname"][cut:]; rec["surname"] = rec["surname"][:cut]
    return rec

# Exact-rate flag assigner (streaming; ensures exact target count over TOTAL_ROWS)
class ExactFlagger:
    def __init__(self, total, count):
        self.total = total; self.left = count; self.idx = 0
    def next(self):
        remain = self.total - self.idx
        take_prob = 0 if remain<=0 else self.left / remain
        self.idx += 1
        if random.random() < take_prob:
            self.left -= 1
            return True
        return False

def main():
    total = TOTAL_ROWS
    inn_flag = ExactFlagger(total, int(total*INN_EMPTY_RATE))
    snils_flag = ExactFlagger(total, int(total*SNILS_EMPTY_RATE))
    docser_flag = ExactFlagger(total, int(total*DOCSER_EMPTY_RATE))

    donors = []
    collisions_target = int(0.01 * total)
    collisions_done = 0

    if OUT_PATH.lower().endswith(".xlsx"):
        if Workbook is None:
            print("openpyxl not available. Please install or use CSV output."); sys.exit(1)
        wb = Workbook(write_only=True)
        ws = wb.create_sheet("subjects")
        ws.append(["source","id_sub","surname","name","secondname","birthdate","INN","SNILS","docnum","docseries"])
        writer = ("xlsx", wb, ws)
    else:
        f = open(OUT_PATH, "w", newline="", encoding="utf-8")
        cw = csv.writer(f); cw.writerow(["source","id_sub","surname","name","secondname","birthdate","INN","SNILS","docnum","docseries"])
        writer = ("csv", f, cw)

    row_idx = 0
    for id_sub, base, k in iter_persons(TOTAL_ROWS, AVG_ROWS_PER_PERSON):
        rows = []
        rows.append({ "source":"GPT","id_sub":id_sub, "surname":base["surname"], "name":base["name"],
                      "secondname":base["secondname"], "birthdate":base["birthdate"],
                      "INN":base["INN"], "SNILS":base["SNILS"], "docnum":base["docnum"], "docseries":base["docseries"] })
        for _ in range(1, k):
            rec = apply_variations(base)
            rows.append({ "source":"GPT","id_sub":id_sub, "surname":rec["surname"], "name":rec["name"],
                          "secondname":rec["secondname"], "birthdate":rec["birthdate"],
                          "INN":rec["INN"], "SNILS":rec["SNILS"], "docnum":rec["docnum"], "docseries":rec["docseries"] })
        for r in rows:
            combine = (random.random() < COMBINE_RATE)

            if combine:
                # Переносим "серия+номер" в docnum (разные форматы),
                # а docseries делаем пустой по флагу или мусорной, чтобы соблюсти правило.
                r["docnum"] = combine_series_num(r["docseries"], r["docnum"])
                if docser_flag.next():
                    r["docseries"] = ""  # учитывается в точной квоте 3%
                else:
                    r["docseries"] = random.choice(garbage_vals)  # мусорное значение
            else:
                # Без объединения: просто отрабатываем пустоту серии по точной квоте.
                if docser_flag.next():
                    r["docseries"] = ""
                # иначе оставляем docseries исходной

            # INN/SNILS пустоты по точным квотам (могут сочетаться между собой)
            if inn_flag.next():
                r["INN"] = ""
            if snils_flag.next():
                r["SNILS"] = ""

            # 1% collisions only on rows with non-empty identifiers
            if collisions_done < collisions_target and (r["INN"]!="" and r["SNILS"]!="" and r["docseries"]!=""):
                if donors and random.random() < collisions_target/total:
                    donor = random.choice(donors)
                    if donor["id_sub"] != r["id_sub"] and (donor["fio"] != (r["surname"], r["name"], r["secondname"])):
                        r["INN"], r["SNILS"], r["docnum"], r["docseries"] = donor["INN"], donor["SNILS"], donor["docnum"], donor["docseries"]
                        collisions_done += 1
                else:
                    donors.append({
                        "id_sub": r["id_sub"],
                        "INN": r["INN"], "SNILS": r["SNILS"], "docnum": r["docnum"], "docseries": r["docseries"],
                        "fio": (r["surname"], r["name"], r["secondname"])
                    })

            if writer[0]=="xlsx":
                writer[2].append([r["source"], r["id_sub"], r["surname"], r["name"], r["secondname"], r["birthdate"], r["INN"], r["SNILS"], r["docnum"], r["docseries"]])
            else:
                writer[2].writerow([r["source"], r["id_sub"], r["surname"], r["name"], r["secondname"], r["birthdate"], r["INN"], r["SNILS"], r["docnum"], r["docseries"]])
            row_idx += 1
            if row_idx >= TOTAL_ROWS: break
        if row_idx >= TOTAL_ROWS: break

    if writer[0]=="xlsx":
        # remove default sheet if exists
        default_title = "Sheet"
        for sht in list(writer[1].sheetnames):
            if sht == default_title:
                del writer[1][sht]
        writer[1].save(OUT_PATH)
    else:
        writer[1].close()

    print("Written:", OUT_PATH, "rows:", row_idx)

if __name__ == "__main__":
    main()

Версия датасета 2 (c id_loan)

In [ ]:
# -*- coding: utf-8 -*-
"""
Generator for 1,000,000-row synthetic credit-subject dataset matching the full spec.

Дополнено колонкой id_loan со следующей логикой:
- большинство займов имеют одного заёмщика (id_loan встречается только с одним id_sub);
- около 5% субъектов участвуют в договорах с несколькими заёмщиками (2–5 id_sub на один id_loan,
  причём ≈80% таких договоров — с двумя заёмщиками);
- небольшая доля (~3% субъектов) попадает в ситуацию "недодубликации": для одного и того же заёмщика
  генерируется 2–3 разных id_loan, хотя по смыслу это один и тот же договор (на несколько id_loan
  приходится один id_sub).

Если нужно изменить доли, правьте константы MULTI_BORROWER_SUBJECT_RATE и ERROR_SPLIT_SUBJECT_RATE.
"""

import random, csv, math, os, sys
from datetime import datetime, timedelta
from typing import Iterator, Tuple, Dict, List

try:
    from openpyxl import Workbook
except Exception:
    Workbook = None  # allow CSV fallback if openpyxl not present

# ---------------- CONFIG ----------------
TOTAL_ROWS = 1_000_000
AVG_ROWS_PER_PERSON = 5.1
OUT_PATH = "credit_subjects_1M.xlsx"  # change to .csv for CSV output
RANDOM_SEED = 20251014

INN_EMPTY_RATE = 0.58
SNILS_EMPTY_RATE = 0.62
DOCSER_EMPTY_RATE = 0.03

# id_loan конфиг
LOAN_ID_BASE = 200_000_000_000
MULTI_BORROWER_SUBJECT_RATE = 0.05  # ~5% субъектов участвуют в договорах с несколькими заёмщиками
ERROR_SPLIT_SUBJECT_RATE = 0.03     # ~3% субъектов имеют 2–3 разных id_loan на одного человека

# Mixing format share (series+number in one field)
COMBINE_RATE = 0.35

random.seed(RANDOM_SEED)

# ---------------- Static resources ----------------
garbage_vals = ["", " ", "\t", "-", "_", "отсутствует", "неизвестно", "Б/Н", "н/д", "N/A", "0", "000000", "1111111"]

male_first_ru = [
    "Александр","Сергей","Дмитрий","Андрей","Алексей","Иван","Николай","Павел","Михаил","Евгений",
    "Владимир","Константин","Григорий","Ярослав","Олег","Тимофей","Кирилл","Фёдор","Роман","Егор"
]
female_first_ru = [
    "Анна","Елена","Ольга","Наталья","Татьяна","Мария","Ирина","Алина","Екатерина","Светлана",
    "Дарья","Виктория","Юлия","Вероника","Ксения","Анастасия","Полина","Людмила","София","Валерия"
]
surnames_ru = [
    "Иванов","Петров","Сидоров","Смирнов","Кузнецов","Попов","Соколов","Лебедев","Козлов","Новиков",
    "Морозов","Ершов","Орлов","Волков","Соловьёв","Егоров","Фролов","Васильев","Зайцев","Карелин",
    "Кореев","Корнеев","Громов","Калинин","Макаров","Никитин","Поляков","Тимофеев","Федоров","Яковлев"
]

male_first_cas = [
    "Алишер","Мурад","Давран","Шерали","Жомарт","Ислом","Шухрат","Улукбек","Абдурахман","Магомед",
    "Мухаммад","Рустам","Фарход","Карим","Нурлан","Сардор","Бахтиёр","Эркин","Азамат","Тимир"
]
female_first_cas = [
    "Азиза","Гульнара","Дилноза","Мехринисо","Ширин","Нилуфар","Саодат","Сания","Зухра","Мунира",
    "Севара","Динара","Айгерим","Айдана","Айгуль","Мадина","Лейла","Гульзира","Гульмира","Жазира"
]
surnames_cas = [
    "Каримов","Нурматов","Ибрагимов","Нуралиев","Хашимов","Рахмонов","Магомедов","Сафаров","Шарипов","Садыков",
    "Умаров","Азимов","Исмаилов","Турсунов","Усманов","Султанов","Юсупов","Нурмухамедов","Тлеубергенов","Алиев"
]

male_first_other = ["John","Michael","David","Daniel","James","Robert","Thomas","William","Richard","Joseph"]
female_first_other = ["Emily","Sophia","Olivia","Emma","Ava","Mia","Chloe","Isabella","Charlotte","Amelia"]
surnames_other = ["Smith","Johnson","Williams","Brown","Jones","Miller","Davis","Wilson","Moore","Taylor"]

patro_roots_m = [
    "Александров","Сергеев","Дмитриев","Андреев","Алексеев","Иванов","Николаев","Павлов","Михайлов","Евгеньев",
    "Владимиров","Константинов","Григорьев","Ярославов","Олегов","Тимофеев","Кириллов","Фёдоров","Романов","Егоров"
]
patro_male = [root + "ич" for root in patro_roots_m]
patro_female = [root + "на" for root in patro_roots_m]

# CХОЖИЕ ФАМИЛИИ
similar_groups = [
    ["Карелин","Кореев","Корнеев"],
    ["Смирнов","Смирнова","Смирной"],
    ["Иванов","Иванова","Иванёва"]
]

# ТРАНСЛИТ
translit_map = {
    "А":"A","Б":"B","В":"V","Г":"G","Д":"D","Е":"E","Ё":"E","Ж":"Zh","З":"Z","И":"I","Й":"Y","К":"K","Л":"L","М":"M",
    "Н":"N","О":"O","П":"P","Р":"R","С":"S","Т":"T","У":"U","Ф":"F","Х":"Kh","Ц":"Ts","Ч":"Ch","Ш":"Sh","Щ":"Sch",
    "Ъ":"","Ы":"Y","Ь":"","Э":"E","Ю":"Yu","Я":"Ya",
    "а":"a","б":"b","в":"v","г":"g","д":"d","е":"e","ё":"e","ж":"zh","з":"z","и":"i","й":"y","к":"k","л":"l","м":"m",
    "н":"n","о":"o","п":"p","р":"r","с":"s","т":"t","у":"u","ф":"f","х":"kh","ц":"ts","ч":"ch","ш":"sh","щ":"sch",
    "ъ":"","ы":"y","ь":"","э":"e","ю":"yu","я":"ya"
}

def to_latin(s: str) -> str:
    return "".join(translit_map.get(ch, ch) for ch in s)

# СОСЕДНИЕ БУКВЫ ПО КЛАВИАТУРЕ
neighbors = {
    "ф":"ы","ы":"фва","в":"ыас","а":"впс","п":"аро","р":"пош","о":"ршл","л":"одж","д":"лор","ж":"лэ","э":"ж",
    "й":"ц","ц":"йу","у":"цк","к":"уе","е":"кн","н":"ег","г":"нш","ш":"гщ","щ":"шз","з":"щх","х":"зъ","ъ":"х",
    "с":"ама","м":"с","и":"т","т":"и","ь":"б","б":"ью","ю":"б"
}

def typo_neighbor(s: str) -> str:
    if not s:
        return s
    idx = random.randrange(len(s))
    ch = s[idx]
    repl = ch
    if ch.lower() in neighbors and neighbors[ch.lower()]:
        repl = random.choice(neighbors[ch.lower()])
        repl = repl.upper() if ch.isupper() else repl
    return s[:idx] + repl + s[idx+1:]

def typo_lev2(s: str) -> str:
    if not s:
        return s
    if random.random() < 0.5:
        return typo_neighbor(typo_neighbor(s))
    else:
        if len(s) > 2 and random.random() < 0.5:
            i = random.randrange(len(s))
            return s[:i] + s[i+1:]
        else:
            i = random.randrange(len(s))
            ch = s[i]
            return s[:i] + ch + s[i:]

def random_date(start_year=1940, end_year=2005):
    d0 = datetime(start_year, 1, 1)
    d1 = datetime(end_year, 12, 31)
    delta = (d1 - d0).days
    return d0 + timedelta(days=random.randrange(delta + 1))

def fmt_date(d: datetime) -> str:
    return d.strftime("%d.%m.%Y")

def perturb_date(d: datetime) -> str:
    choice = random.random()
    if choice < 0.15:
        return random.choice(garbage_vals)
    if choice < 0.45:
        dd = random.choice([-2,-1,1,2])
        return fmt_date(d + timedelta(days=dd))
    elif choice < 0.8:
        mm = random.choice([-2,-1,1,2])
        y = d.year
        m = max(1, min(12, d.month + mm))
        day = min(d.day, 28)
        return f"{day:02d}.{m:02d}.{y:04d}"
    else:
        yy = random.choice([-1,1])
        day = min(d.day, 28)
        return f"{day:02d}.{d.month:02d}.{d.year+yy:04d}"

def gen_inn():
    return "".join(str(random.randrange(10)) for _ in range(12))

def gen_snils():
    nums = [random.randrange(10) for _ in range(11)]
    return f"{nums[0]}{nums[1]}{nums[2]}-{nums[3]}{nums[4]}{nums[5]}-{nums[6]}{nums[7]}{nums[8]} {nums[9]}{nums[10]}"

def gen_passport_pair():
    series = f"{random.randrange(1000,10000)}"
    number = f"{random.randrange(100000,1000000)}"
    return series, number

def combine_series_num(series, num):
    options = [f"{series} {num}", f"{series}{num}", f"{series}-{num}", f"{series}/{num}"]
    return random.choice(options)

def sample_record_count(group):
    r = random.random()
    if group == "east":
        return 3 if r < 0.15 else 4 if r < 0.40 else 5 if r < 0.70 else 6 if r < 0.90 else 7
    elif group == "central":
        return 4 if r < 0.10 else 5 if r < 0.30 else 6 if r < 0.60 else 7 if r < 0.85 else 8 if r < 0.95 else 9
    else:
        return 5 if r < 0.10 else 6 if r < 0.35 else 7 if r < 0.65 else 8 if r < 0.85 else 9 if r < 0.95 else 10

def make_feminine(last):
    if not last.endswith("а") and last.endswith(("ов","ев","ин")):
        return last + "а"
    return last

def iter_persons(total_rows: int, avg_per_person: float):
    persons_target = int(round(total_rows / avg_per_person))
    east_q = int(round(persons_target * 0.80))
    central_q = int(round(persons_target * 0.19))
    other_q = persons_target - east_q - central_q

    male_q = persons_target // 2
    female_q = persons_target - male_q

    def split_gender(q, m_rem, f_rem):
        m = min(m_rem, q // 2)
        f = min(f_rem, q - m)
        if f < q - m:
            need = (q - m) - f
            m = min(m + need, m_rem)
            f = q - m
        return m, f

    m_rem, f_rem = male_q, female_q
    e_m, e_f = split_gender(east_q, m_rem, f_rem); m_rem -= e_m; f_rem -= e_f
    c_m, c_f = split_gender(central_q, m_rem, f_rem); m_rem -= c_m; f_rem -= c_f
    o_m, o_f = split_gender(other_q, m_rem, f_rem); m_rem -= o_m; f_rem -= o_f

    meta: List[Tuple[str,str]] = (
        [("east","M")] * e_m +
        [("east","F")] * e_f +
        [("central","M")] * c_m +
        [("central","F")] * c_f +
        [("other","M")] * o_m +
        [("other","F")] * o_f
    )
    random.shuffle(meta)

    ks = [sample_record_count(g) for g,_ in meta]
    delta = total_rows - sum(ks)

    idx_e = [i for i,(g,_) in enumerate(meta) if g=="east"]
    idx_c = [i for i,(g,_) in enumerate(meta) if g=="central"]
    idx_o = [i for i,(g,_) in enumerate(meta) if g=="other"]

    while delta != 0:
        if delta > 0:
            pool = idx_c + idx_o + idx_e
            random.shuffle(pool)
            for i in pool:
                if ks[i] < 15:
                    ks[i] += 1
                    delta -= 1
                    if delta==0:
                        break
        else:
            pool = idx_e + idx_c + idx_o
            random.shuffle(pool)
            for i in pool:
                if ks[i] > 1:
                    ks[i] -= 1
                    delta += 1
                    if delta==0:
                        break

    base_id = 90_000_000_000
    for pid, ((gid,sex), k) in enumerate(zip(meta, ks)):
        if gid == "east":
            first = random.choice(male_first_ru if sex=="M" else female_first_ru)
            last = random.choice(surnames_ru)
            if sex == "F":
                last = make_feminine(last)
            patronymic = random.choice(patro_male if sex=="M" else patro_female)
        elif gid == "central":
            first = random.choice(male_first_cas if sex=="M" else female_first_cas)
            last = random.choice(surnames_cas)
            patronymic = "" if random.random() < 0.7 else random.choice(patro_male if sex=="M" else patro_female)
        else:
            first = random.choice(male_first_other if sex=="M" else female_first_other)
            last = random.choice(surnames_other)
            patronymic = ""

        married_surnames: List[str] = []
        if sex == "F":
            src_pool = surnames_ru if gid=="east" else (surnames_cas if gid=="central" else surnames_other)
            max_changes = 2 if random.random() < 0.35 else 1
            if gid=="east" and random.random()<0.30:
                married_surnames = [make_feminine(random.choice(src_pool)) for _ in range(max_changes)]
            elif gid=="central" and random.random()<0.20:
                married_surnames = [random.choice(src_pool) for _ in range(max_changes)]
            elif gid=="other" and random.random()<0.10:
                married_surnames = [random.choice(src_pool) for _ in range(max_changes)]
            married_surnames = list(dict.fromkeys(married_surnames))

        bdate = random_date()
        inn = gen_inn()
        snils = gen_snils()
        ds, dn = gen_passport_pair()

        base = {
            "group": gid,
            "gender": sex,
            "surname": last,
            "name": first,
            "secondname": patronymic,
            "birthdate": fmt_date(bdate),
            "INN": inn,
            "SNILS": snils,
            "docseries": ds,
            "docnum": dn,
            "married_surnames": married_surnames,
        }
        yield base_id + pid, base, k

def random_name_permutation(surname, name, secondname):
    if random.random() < 0.5:
        return name, surname, secondname
    else:
        return (secondname or surname), name, (surname if secondname else "")

identifier_fields = {"INN","SNILS","docnum","docseries"}
field_weights = {
    "surname":0.15,"birthdate":0.15,"name":0.10,"secondname":0.10,
    "INN":0.15,"SNILS":0.15,"docnum":0.075,"docseries":0.075
}

def weighted_field_choices(k: int):
    fields = list(field_weights.keys())
    probs = [field_weights[f] for f in fields]
    chosen: List[str] = []
    while len(chosen) < k:
        cand = random.choices(fields, weights=probs, k=1)[0]
        if cand not in chosen:
            chosen.append(cand)
        if len(chosen) == len(fields):
            break
    chosen_ids = [f for f in chosen if f in identifier_fields]
    if len(chosen_ids) > 1:
        keep = random.choice(chosen_ids)
        chosen = [f for f in chosen if f not in identifier_fields or f==keep]
    return chosen

def apply_variations(base):
    gid = base["group"]
    sex = base["gender"]
    rec = {k: base[k] for k in ["surname","name","secondname","birthdate","INN","SNILS","docnum","docseries"]}

    multi = (random.random() < 0.20)
    m = 1 if not multi else random.randint(2,4)
    change_fields = set(weighted_field_choices(m))

    cyr_lat_issue = (gid != "east" and random.random() < 0.8)
    secondname_issue = (gid != "east" and random.random() < 0.8)

    for f in list(change_fields):
        if f in ["surname","name","secondname"]:
            val = rec[f]
            garbage_boost = 0.05 + (0.05 if gid != "east" else 0.0)
            if random.random() < garbage_boost:
                rec[f] = random.choice(garbage_vals)
                continue
            if f=="surname" and random.random() < 0.15:
                rec[f] = random.choice(random.choice(similar_groups))
            else:
                rec[f] = typo_neighbor(val) if random.random()<0.6 else typo_lev2(val)
        elif f=="birthdate":
            bd = datetime.strptime(base["birthdate"], "%d.%m.%Y")
            rec["birthdate"] = perturb_date(bd)
        elif f in identifier_fields:
            if random.random() < 0.15:
                g = random.choice(garbage_vals[1:]) if f=="docnum" else random.choice(garbage_vals)
                if f=="docnum" and g.strip()=="":
                    g = "000000"
                rec[f] = g
            else:
                s = rec[f]
                digits_idx = [i for i,ch in enumerate(s) if ch.isdigit()]
                if digits_idx:
                    edits = 2 if random.random() < 0.4 else 1
                    s_list = list(s)
                    for _ in range(edits):
                        i = random.choice(digits_idx)
                        s_list[i] = str(random.randrange(10))
                    rec[f] = "".join(s_list)

    if random.random() < 0.10:
        s, n, p = random_name_permutation(rec["surname"], rec["name"], rec["secondname"])
        rec["surname"], rec["name"], rec["secondname"] = s, n, p

    if sex=="F" and base.get("married_surnames"):
        if "surname" in change_fields or random.random() < 0.20:
            rec["surname"] = random.choice(base["married_surnames"])

    if cyr_lat_issue:
        for k in ["surname","name"] + (["secondname"] if rec["secondname"] else []):
            if rec[k] and random.random() < 0.5:
                rec[k] = to_latin(rec[k])

    if secondname_issue and random.random() < 0.5:
        rec["secondname"] = random.choice([rec["secondname"], "", random.choice(garbage_vals)])

    if gid!="east" and random.random() < 0.08:
        if rec["surname"] and rec["name"]:
            if random.random() < 0.5:
                rec["name"] = rec["name"] + rec["surname"][:2]
            else:
                if len(rec["surname"]) > 6:
                    cut = random.randint(3, len(rec["surname"]) - 3)
                    rec["secondname"] = rec["surname"][cut:]
                    rec["surname"] = rec["surname"][:cut]

    return rec

# -------- Loan mapping helpers --------

def build_multi_borrower_clusters(person_count: int, multi_share: float) -> List[List[int]]:
    """Создаёт группы индексов субъектов, которые будут делить один и тот же id_loan."""
    total_multi_subjects = int(round(person_count * multi_share))
    if total_multi_subjects == 0:
        return []

    indices = list(range(person_count))
    random.shuffle(indices)
    multi_subjects = indices[:total_multi_subjects]

    random.shuffle(multi_subjects)
    target_pair_subjects = int(round(total_multi_subjects * 0.80))
    if target_pair_subjects % 2 == 1:
        target_pair_subjects -= 1
    num_pairs = target_pair_subjects // 2

    clusters: List[List[int]] = []
    idx_pos = 0

    # Пары
    for _ in range(num_pairs):
        if idx_pos + 1 >= len(multi_subjects):
            break
        s1 = multi_subjects[idx_pos]
        s2 = multi_subjects[idx_pos + 1]
        clusters.append([s1, s2])
        idx_pos += 2

    remaining = multi_subjects[idx_pos:]

    # Кластеры 3–5 заёмщиков
    while remaining:
        remain = len(remaining)
        if remain <= 5:
            clusters.append(remaining)
            break
        max_size = min(5, remain - 3)
        size = random.randint(3, max_size)
        clusters.append(remaining[:size])
        remaining = remaining[size:]

    return clusters

def plan_loan_mapping(person_count: int) -> List[List[int]]:
    """Возвращает список: для каждого индекса субъекта список id_loan, с которыми он связан."""
    loan_ids_by_subject: List[List[int]] = [[] for _ in range(person_count)]
    next_loan_id = LOAN_ID_BASE

    # 1) Многозаёмные договоры: один id_loan на несколько id_sub
    clusters = build_multi_borrower_clusters(person_count, MULTI_BORROWER_SUBJECT_RATE)
    for cluster in clusters:
        loan_id = next_loan_id
        next_loan_id += 1
        for subj_idx in cluster:
            loan_ids_by_subject[subj_idx].append(loan_id)

    # 2) Всем остальным субъектам выдаём по одному "одиночному" займу
    for subj_idx in range(person_count):
        if not loan_ids_by_subject[subj_idx]:
            loan_ids_by_subject[subj_idx].append(next_loan_id)
            next_loan_id += 1

    # 3) Ошибочная "недодубликация": для части субъектов делаем 2–3 разных id_loan
    candidates = [i for i in range(person_count) if len(loan_ids_by_subject[i]) == 1]
    num_error_subjects = int(round(person_count * ERROR_SPLIT_SUBJECT_RATE))
    num_error_subjects = min(num_error_subjects, len(candidates))

    random.shuffle(candidates)
    error_subjects = candidates[:num_error_subjects]

    for subj_idx in error_subjects:
        total_loans_for_subj = 2 if random.random() < 0.8 else 3
        # уже есть 1 id_loan, добавляем ещё (total_loans_for_subj - 1)
        for _ in range(total_loans_for_subj - 1):
            loan_ids_by_subject[subj_idx].append(next_loan_id)
            next_loan_id += 1

    return loan_ids_by_subject

# Exact-rate flag assigner (streaming; ensures exact target count over TOTAL_ROWS)
class ExactFlagger:
    def __init__(self, total, count):
        self.total = total
        self.left = count
        self.idx = 0

    def next(self):
        remain = self.total - self.idx
        take_prob = 0 if remain <= 0 else self.left / remain
        self.idx += 1
        if random.random() < take_prob:
            self.left -= 1
            return True
        return False

def main():
    total = TOTAL_ROWS
    inn_flag = ExactFlagger(total, int(total * INN_EMPTY_RATE))
    snils_flag = ExactFlagger(total, int(total * SNILS_EMPTY_RATE))
    docser_flag = ExactFlagger(total, int(total * DOCSER_EMPTY_RATE))

    # Планируем распределение id_loan на уровне субъектов
    person_count = int(round(TOTAL_ROWS / AVG_ROWS_PER_PERSON))
    loan_ids_by_subject = plan_loan_mapping(person_count)

    donors = []
    collisions_target = int(0.01 * total)
    collisions_done = 0

    if OUT_PATH.lower().endswith(".xlsx"):
        if Workbook is None:
            print("openpyxl not available. Please install or use CSV output.")
            sys.exit(1)
        wb = Workbook(write_only=True)
        ws = wb.create_sheet("subjects")
        ws.append([
            "source","id_sub","id_loan","surname","name","secondname",
            "birthdate","INN","SNILS","docnum","docseries"
        ])
        writer = ("xlsx", wb, ws)
    else:
        f = open(OUT_PATH, "w", newline="", encoding="utf-8")
        cw = csv.writer(f)
        cw.writerow([
            "source","id_sub","id_loan","surname","name","secondname",
            "birthdate","INN","SNILS","docnum","docseries"
        ])
        writer = ("csv", f, cw)

    row_idx = 0

    for subj_idx, (id_sub, base, k) in enumerate(iter_persons(TOTAL_ROWS, AVG_ROWS_PER_PERSON)):
        if subj_idx >= len(loan_ids_by_subject):
            # На всякий случай (не должно происходить)
            loan_ids_for_subject = [LOAN_ID_BASE + subj_idx]
        else:
            loan_ids_for_subject = loan_ids_by_subject[subj_idx]

        num_loans_for_subject = len(loan_ids_for_subject)

        # Для данного субъекта и его k строк:
        #  - гарантируем хотя бы по одной строке на каждый id_loan
        #  - оставшиеся строки размазываем по этим id_loan случайно
        assigned_loan_ids: List[int] = []
        for lid in loan_ids_for_subject:
            assigned_loan_ids.append(lid)
        for _ in range(k - num_loans_for_subject):
            assigned_loan_ids.append(random.choice(loan_ids_for_subject))
        random.shuffle(assigned_loan_ids)

        rows = []
        # базовая "чистая" запись
        rows.append({
            "source": "GPT",
            "id_sub": id_sub,
            "id_loan": assigned_loan_ids[0],
            "surname": base["surname"],
            "name": base["name"],
            "secondname": base["secondname"],
            "birthdate": base["birthdate"],
            "INN": base["INN"],
            "SNILS": base["SNILS"],
            "docnum": base["docnum"],
            "docseries": base["docseries"],
        })

        # разнородные записи
        for idx_local in range(1, k):
            rec = apply_variations(base)
            rows.append({
                "source": "GPT",
                "id_sub": id_sub,
                "id_loan": assigned_loan_ids[idx_local],
                "surname": rec["surname"],
                "name": rec["name"],
                "secondname": rec["secondname"],
                "birthdate": rec["birthdate"],
                "INN": rec["INN"],
                "SNILS": rec["SNILS"],
                "docnum": rec["docnum"],
                "docseries": rec["docseries"],
            })

        for r in rows:
            combine = (random.random() < COMBINE_RATE)
            if combine:
                # доклеиваем серию к номеру
                r["docnum"] = combine_series_num(r["docseries"], r["docnum"])
                if docser_flag.next():
                    r["docseries"] = ""
                else:
                    r["docseries"] = random.choice(garbage_vals)
            else:
                if docser_flag.next():
                    r["docseries"] = ""

            # INN / SNILS пустые по точным квотам
            if inn_flag.next():
                r["INN"] = ""
            if snils_flag.next():
                r["SNILS"] = ""

            # 1% коллизий идентификаторов при разных ФИО
            if (collisions_done < collisions_target and
                (r["INN"] != "" and r["SNILS"] != "" and r["docseries"] != "")):
                if donors and random.random() < collisions_target / total:
                    donor = random.choice(donors)
                    if donor["id_sub"] != r["id_sub"] and (
                        donor["fio"] != (r["surname"], r["name"], r["secondname"])
                    ):
                        r["INN"], r["SNILS"], r["docnum"], r["docseries"] = (
                            donor["INN"], donor["SNILS"], donor["docnum"], donor["docseries"]
                        )
                        collisions_done += 1
                else:
                    donors.append({
                        "id_sub": r["id_sub"],
                        "INN": r["INN"],
                        "SNILS": r["SNILS"],
                        "docnum": r["docnum"],
                        "docseries": r["docseries"],
                        "fio": (r["surname"], r["name"], r["secondname"]),
                    })

            if writer[0] == "xlsx":
                writer[2].append([
                    r["source"], r["id_sub"], r["id_loan"], r["surname"], r["name"], r["secondname"],
                    r["birthdate"], r["INN"], r["SNILS"], r["docnum"], r["docseries"]
                ])
            else:
                writer[2].writerow([
                    r["source"], r["id_sub"], r["id_loan"], r["surname"], r["name"], r["secondname"],
                    r["birthdate"], r["INN"], r["SNILS"], r["docnum"], r["docseries"]
                ])

            row_idx += 1
            if row_idx >= TOTAL_ROWS:
                break

        if row_idx >= TOTAL_ROWS:
            break

    if writer[0] == "xlsx":
        # remove default sheet if exists
        default_title = "Sheet"
        for sht in list(writer[1].sheetnames):
            if sht == default_title:
                del writer[1][sht]
        writer[1].save(OUT_PATH)
    else:
        writer[1].close()

    print("Written:", OUT_PATH, "rows:", row_idx)

if __name__ == "__main__":
    main()

Written: credit_subjects_1M.xlsx rows: 1000000


In [ ]:
import pandas as pd
e = pd.read_excel("credit_subjects_1M.xlsx")
e.head()

,source,id_sub,id_loan,surname,name,secondname,birthdate,INN,SNILS,docnum,docseries
0,GPT,90000000000,200000004413,Фролов,Евгений,Андреевич,11.03.1951,115662292324,NaN,7189231949,-
1,GPT,90000000000,200000004413,Фршлоы,Евгений,Андреевич,11.03.1951,NaN,674-266-140 57,231949,7189
2,GPT,90000000000,200000004413,Фролов,Евгений,Андреевич,11.04.1951,Б/Н,674-266-140 57,7189-231949,-
3,GPT,90000000001,200000004414,Садыков,Мухаммад,NaN,28.06.2004,NaN,NaN,724764,4831
4,GPT,90000000001,200000004414,Sadykov,NaN,NaN,28.06.2004,NaN,NaN,4831724764,1111111


Baseline-решение (Spark GraphX)

1. Загрузка Excel в Spark DataFrame

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("credit_subjects_graph_baseline") \
    .getOrCreate()

# 1) читаем Excel в pandas
pdf = pd.read_excel("credit_subjects_1M.xlsx")

# 2) в Spark
df = spark.createDataFrame(pdf)

2. Подготовка данных: row_id, нормализация ключей

Код добавляет технический ID строки `row_id`, а затем строит `df_norm`, где **чистит и нормализует идентификаторы и ФИО**: из `INN` и `SNILS` оставляет только цифры, из серии и номера документа — только буквы/цифры и склеивает их в паспортный ключ `doc_key`; ФИО приводит к нижнему регистру и убирает лишние пробелы, дату рождения — триммирует, после чего собирает ключ `fio_dob_key = "фамилия|имя|отчество|дата"` для матчинга записей и поиска дубликатов.

In [ ]:
from pyspark.sql import functions as F

# Добавляем row_id для вершин графа
df = df.withColumn("row_id", F.monotonically_increasing_id().cast("long"))

# Нормализация идентификаторов и ФИО
df_norm = (
    df
    # чистим ИНН/СНИЛС/паспорт до цифр
    .withColumn("inn_norm",   F.regexp_replace(F.col("INN"),   "[^0-9]", ""))
    .withColumn("snils_norm", F.regexp_replace(F.col("SNILS"), "[^0-9]", ""))
    .withColumn("docseries_norm", F.regexp_replace(F.col("docseries"), "[^0-9A-Za-zА-Яа-я]", ""))
    .withColumn("docnum_norm",    F.regexp_replace(F.col("docnum"),    "[^0-9A-Za-zА-Яа-я]", ""))

    # объединённый паспортный ключ (серия+номер)
    .withColumn("doc_key", F.concat_ws("", "docseries_norm", "docnum_norm"))

    # нормализуем ФИО (пока только lower+trim, без транслита для простоты)
    .withColumn("surname_norm",    F.lower(F.trim(F.col("surname"))))
    .withColumn("name_norm",       F.lower(F.trim(F.col("name"))))
    .withColumn("secondname_norm", F.lower(F.trim(F.col("secondname"))))

    # дату пока берём как есть строку (на синтетике формат один)
    .withColumn("birthdate_norm", F.trim(F.col("birthdate")))

    # ключ ФИО+дата рождения
    .withColumn(
        "fio_dob_key",
        F.concat_ws("|", "surname_norm", "name_norm", "secondname_norm", "birthdate_norm")
    )
)


3. Построение графа и connected components (ручная реализация на PySpark)

3.1. UDF для генерации пар в блоке

Функция сначала задаёт схему `pairs_schema` — это массив структур вида `{src: long, dst: long}` для хранения пар вершин, а затем определяет UDF `all_pairs`, которая принимает массив `ids`, приводя элементы к `int`, и **генерирует все уникальные пары индексов**: для каждой комбинации `i < j` добавляет в результат структуру `{"src": ids[i], "dst": ids[j]}` (если значения не совпадают), возвращая список таких пар; пустой или `None`-вход даёт пустой список.

In [ ]:
from pyspark.sql.types import ArrayType, StructType, StructField, LongType
from pyspark.sql import functions as F

pairs_schema = ArrayType(
    StructType([
        StructField("src", LongType(), nullable=False),
        StructField("dst", LongType(), nullable=False),
    ])
)

@F.udf(returnType=pairs_schema)
def all_pairs(ids):
    if ids is None:
        return []
    ids = [int(x) for x in ids]
    res = []
    n = len(ids)
    for i in range(n):
        for j in range(i + 1, n):
            # проверка на всякий
            if ids[i] != ids[j]:
                res.append({"src": ids[i], "dst": ids[j]})
    return res


3.2. Генерация рёбер по разным ключам

Функция `edges_from_key` по заданному ключу строит рёбра графа между записями, которые разделяют одно и то же значение ключа: она берёт `row_id` и `key_col`, отфильтровывает пустые ключи, группирует по `key_col` и собирает список `row_id` в блоки, отсеивает блоки с одной строкой и слишком большие (больше `max_block_size`), затем с помощью UDF `all_pairs` для каждого блока генерирует все пары записей и разворачивает их в строки, возвращая датафрейм со столбцами `src`, `dst` и константным `edge_type`, который помечает тип ребра (но на `connectedComponents` не влияет).

In [ ]:
def edges_from_key(df, key_col, edge_type, max_block_size=50):
    """
    df       — DataFrame с row_id и key_col
    key_col  — имя колонки (строка)
    edge_type — метка типа ребра (string), на connectedComponents не влияет
    max_block_size — чтобы не взрывались огромные блоки (порог можно менять)
    """
    return (
        df
        .select("row_id", key_col)
        .where(F.col(key_col).isNotNull() & (F.length(F.col(key_col)) > 0))
        .groupBy(key_col)
        .agg(F.collect_list("row_id").alias("rows"))
        .where(F.size("rows") > 1)
        .where(F.size("rows") <= max_block_size)  # отсечение монстров
        .withColumn("pairs", all_pairs(F.col("rows")))
        .select(F.explode("pairs").alias("p"))
        .select(
            F.col("p.src").alias("src"),
            F.col("p.dst").alias("dst"),
            F.lit(edge_type).alias("edge_type")
        )
    )


3.2.2. Формирование рёбр и вершин

Код строит рёбра и вершины графа: сначала для каждого типа ключа (`inn_norm`, `snils_norm`, `doc_key`, `fio_dob_key`) вызывается `edges_from_key`, получая четыре датафрейма рёбер с пометкой типа (`"inn"`, `"snils"`, `"passport"`, `"fio_dob"`), затем они **объединяются в единый датафрейм `edges`**, дубликаты пар (`src`, `dst`) удаляются и выводится их количество. После этого формируется датафрейм **вершин** `vertices`, где `row_id` переименовывается в `id`, а к нему добавляются бизнес-поля (`id_sub`, `id_loan`, ФИО, дата рождения, ИНН, СНИЛС, реквизиты документа), и также печатается количество вершин.

In [ ]:
# рёбра по разным ключам
edges_inn   = edges_from_key(df_norm, "inn_norm",   "inn")
edges_snils = edges_from_key(df_norm, "snils_norm", "snils")
edges_doc   = edges_from_key(df_norm, "doc_key",    "passport")
edges_fio   = edges_from_key(df_norm, "fio_dob_key","fio_dob")

# объединяем в один DataFrame edges
edges = (
    edges_inn
    .union(edges_snils)
    .union(edges_doc)
    .union(edges_fio)
    .dropDuplicates(["src", "dst"])
)

print("edges count =", edges.count())

vertices = df_norm.select(
    F.col("row_id").alias("id"),
    "id_sub", "id_loan",
    "surname", "name", "secondname",
    "birthdate", "INN", "SNILS", "docnum", "docseries"
)

print("vertices count =", vertices.count())

edges count = 1418143
vertices count = 1000000


3.3. Реализация connected components на PySpark

Код вручную считает компоненты связности через итеративное распространение минимальной метки. Сначала из `edges` строится **неориентированный граф** `edges_undir`: для каждого ребра добавляется зеркальное (`src↔dst`), затем удаляются дубликаты; `edges_undir` и `vertices` один раз **persist**-ятся, чтобы оборвать lineage и разгрузить план. Далее инициализируются метки `labels`: для каждой вершины `id` её компонентой `comp` считается она же (тривиальные компоненты). В цикле из `max_iters` итераций считается `neighbor_min`: для каждой вершины-источника берётся минимальная компонентная метка её соседей (`min(comp)` по `dst`), после чего формируются новые метки `new_labels`, где `comp` обновляется до `least(comp, min_comp_neighbor)` (если соседей нет — оставляем старую метку). На каждой итерации новый датафрейм меток **persist**-ится, материализуется через `count()`, а старый `labels.unpersist()` — так ограничивается рост DAG и потребление памяти. В конце полученные стабильные `labels` снова `persist`-ятся и джойнятся к `vertices_p`, формируя итоговый датафрейм `df_cc`, где у каждой вершины появляется предсказанный идентификатор компоненты `id_sub_pred`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

# 1) неориентированный граф (как было)
edges_undir = (
    edges.select("src", "dst")
    .union(
        edges.select(
            F.col("dst").alias("src"),
            F.col("src").alias("dst")
        )
    )
    .dropDuplicates(["src", "dst"])
)

# Персистим один раз, чтобы оборвать lineage
edges_undir = edges_undir.persist(StorageLevel.MEMORY_AND_DISK)
edges_undir.count()  # материализуем

vertices_p = vertices.persist(StorageLevel.MEMORY_AND_DISK)
vertices_p.count()

# 2) Инициализация: comp = id
labels = vertices_p.select(
    F.col("id").alias("id"),
    F.col("id").alias("comp")
)
labels = labels.persist(StorageLevel.MEMORY_AND_DISK)
labels.count()

# 3) Фиксированные итерации без cache, с аккуратным unpersist
max_iters = 6  # начнём с 5–6, чтобы не насиловать heap

for it in range(max_iters):
    print(f"Iter {it}")

    neighbor_min = (
        edges_undir
        .join(labels, edges_undir.dst == labels.id)
        .groupBy("src")
        .agg(F.min("comp").alias("min_comp_neighbor"))
        .select(
            F.col("src").alias("id"),
            "min_comp_neighbor"
        )
    )

    new_labels = (
        labels
        .join(neighbor_min, on="id", how="left")
        .select(
            "id",
            F.when(
                F.col("min_comp_neighbor").isNull(),
                F.col("comp")
            ).otherwise(
                F.least(F.col("comp"), F.col("min_comp_neighbor"))
            ).alias("comp")
        )
    )

    # избегаем роста плана: persist + материализация + unpersist старого
    new_labels = new_labels.persist(StorageLevel.MEMORY_AND_DISK)
    new_labels.count()

    labels.unpersist()
    labels = new_labels

# финальные labels
labels = labels.persist(StorageLevel.MEMORY_AND_DISK)
labels.count()

df_cc = (
    vertices_p.join(labels, on="id", how="left")
    .withColumnRenamed("comp", "id_sub_pred")
)

Iter 0
Iter 1
Iter 2
Iter 3
Iter 4
Iter 5


4. Метрики микрокластеризации субъектов

Код считает **попарные метрики качества кластеризации**: берёт `df_cc` и оставляет `id`, истинный субъект `id_sub` и предсказанный кластер `id_sub_pred`; далее считает, сколько записей попало в каждую комбинацию (`кластер, истинный субъект`) и по этим частотам вычисляет число **True Positive** пар `tp` как сумму `cnt * (cnt - 1) / 2` — то есть пар строк, которые и в истине, и в предсказании лежат вместе. Отдельно считаются размеры предсказанных кластеров (`n_c`) и истинных субъектов (`n_s`), из них получаются общее число пар в предсказанных кластерах `pred_pairs` и в истинных `true_pairs`. Затем из этих величин выводятся **FP** (`pred_pairs - tp`) и **FN** (`true_pairs - tp`), и на их основе считаются **precision = tp / pred_pairs**, **recall = tp / true_pairs** и **F1** (гармоническое среднее precision и recall); в конце всё красиво печатается в консоль.

In [ ]:
from pyspark.sql import functions as F

df_eval = df_cc.select("id", "id_sub", "id_sub_pred")

# Кол-во записей на (кластер, истинный субъект)
cluster_sub_counts = (
    df_eval
    .groupBy("id_sub_pred", "id_sub")
    .agg(F.count("*").alias("cnt"))
)

# True Positive: пары строк, которые и в предсказанном, и в истинном кластере вместе
tp = cluster_sub_counts.select(
    F.sum(F.col("cnt") * (F.col("cnt") - 1) / 2).alias("tp")
).first()["tp"]

# Размеры предсказанных кластеров
cluster_sizes = (
    df_eval
    .groupBy("id_sub_pred")
    .agg(F.count("*").alias("n_c"))
)

pred_pairs = cluster_sizes.select(
    F.sum(F.col("n_c") * (F.col("n_c") - 1) / 2).alias("pred_pairs")
).first()["pred_pairs"]

# Истинные размеры субъектов
sub_sizes = (
    df_eval
    .groupBy("id_sub")
    .agg(F.count("*").alias("n_s"))
)

true_pairs = sub_sizes.select(
    F.sum(F.col("n_s") * (F.col("n_s") - 1) / 2).alias("true_pairs")
).first()["true_pairs"]

tp = float(tp or 0.0)
pred_pairs = float(pred_pairs or 0.0)
true_pairs = float(true_pairs or 0.0)

fn = true_pairs - tp
fp = pred_pairs - tp

precision = tp / pred_pairs if pred_pairs > 0 else 0.0
recall    = tp / true_pairs if true_pairs > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("Pairwise metrics:")
print(f"  TP = {tp:,.0f}")
print(f"  FP = {fp:,.0f}")
print(f"  FN = {fn:,.0f}")
print(f"  precision = {precision:.4f}")
print(f"  recall    = {recall:.4f}")
print(f"  F1        = {f1:.4f}")


Pairwise metrics:
  TP = 1,803,209
  FP = 81,312
  FN = 425,655
  precision = 0.9569
  recall    = 0.8090
  F1        = 0.8768


## 5. Анализ baseline и направления улучшений

**Результат baseline (pairwise-метрики по `id_sub` как истине):** precision = 0.957, recall = 0.809, F1 = 0.877 (TP ≈ 1.80 млн, FP ≈ 81 тыс., FN ≈ 426 тыс. пар).

**Почему высокая precision, но ниже recall.** Рёбра строятся только по точному совпадению ключей (ИНН, СНИЛС, серия+номер паспорта, ФИО+дата рождения). Записи, в которых испорчено сразу несколько ключей (опечатки, транслит, перепутанные части ФИО, ошибки в дате, мусорные значения, слитные/раздельные серия и номер), остаются несвязанными. Это даёт основную часть FN.

**Источники ложных объединений (FP).**
- Коллизии идентификаторов у разных людей (~1% по спецификации): одного совпадения по ИНН/СНИЛС/паспорту достаточно для ребра, а компоненты связности транзитивно «склеивают» кластеры.
- Мусорные значения (`0`, `000000`, `1111111` и т.п.) после нормализации могут становиться ключами блока.

**Ограничения текущей реализации.**
- Распространение минимальной метки идёт фиксированные 6 итераций, без проверки сходимости.
- `id_loan` в матчинге не используется, хотя в версии 2 датасета добавлен именно для улучшения дедупликации.
- ФИО нормализуется только `lower` + `trim`, без транслитерации и сортировки токенов.

**Направления улучшений.**
1. Нормализация: чистка мусорных значений до NULL, транслитерация в единый алфавит, сортировка токенов ФИО, чтение Excel/CSV с `dtype=str`.
2. Fuzzy matching: блокировка (дата рождения, первые буквы, фонетический ключ) + Levenshtein / Jaro-Winkler внутри блоков.
3. Weighted graph: вес ребра = сумма вкладов совпавших признаков, ребро только при превышении порога; это защищает от коллизий идентификаторов.
4. Probabilistic record linkage (Fellegi–Sunter): веса признаков из m- и u-вероятностей.
5. ML-классификация пар: признаки сходства по полям + градиентный бустинг, затем кластеризация по предсказанным парам.
6. Использовать `id_loan` как дополнительный сигнал (только вместе с похожестью ФИО).
7. Проверка сходимости connected components (или GraphFrames / `scipy.sparse.csgraph`).
